SETUP

In [2]:
import os
import time
import uuid
import json
from dotenv import load_dotenv

In [3]:
from portkey_ai import Portkey, createHeaders, PORTKEY_GATEWAY_URL

In [5]:
load_dotenv(dotenv_path="../.env")
PORTKEY_API_KEY=os.getenv("PORTKEY_API_KEY","oTp81rbDw30yq5fK8FEDAu6LoQLM")
portkey= Portkey(api_key=PORTKEY_API_KEY)

SETUP VIRTUAL KEYS AND SLUGS

In [18]:
GROQ_SLUG="gkey"
GROQ_MODEL=f"@{GROQ_SLUG}/openai/gpt-oss-120b"

GROQ_SLUG_2="gkey"
GROQ_MODEL_2=f"@{GROQ_SLUG_2}/llama-3.1-8b-instant"

GROQ_API_KEY=os.getenv("GROQ_API_KEY")

In [19]:
print("Setup complete!")
print(f"  Portkey API Key : {'OK' if PORTKEY_API_KEY else 'MISSING'}")
print(f"  Groq slug       : {GROQ_SLUG}")
print(f"  Groq model ref  : {GROQ_MODEL}")
print(f"  Groq slug 2     : {GROQ_SLUG_2}")
print(f"  Small model ref : {GROQ_MODEL_2}")
print(f"\nPortkey Gateway : {PORTKEY_GATEWAY_URL}")

Setup complete!
  Portkey API Key : OK
  Groq slug       : gkey
  Groq model ref  : @gkey/openai/gpt-oss-120b
  Groq slug 2     : gkey
  Small model ref : @gkey/llama-3.1-8b-instant

Portkey Gateway : https://api.portkey.ai/v1


HELPER FUNCTION

In [20]:
## To print the title
def section(title):
    print(f"\n{"="*65}")
    print(f"{title}")
    print(f"{"="*65}")

section("Abhay")



Abhay


RESPONSE

In [21]:
def show(q, answer, ms, label=""):
    bar = chr(9472) * 62
    print(f"\n{bar}")
    print(f"Q: {q}")
    print(f"A: {answer[:260]}{'...' if len(answer) > 260 else ''}")
    note = f" | {label}" if label else ""
    print(f"⏱  {ms:.0f}ms{note}")
    print(bar)

In [22]:
## it is just to print god nothing much
show("why is the sky blue ?","nature",5,"god")


──────────────────────────────────────────────────────────────
Q: why is the sky blue ?
A: nature
⏱  5ms | god
──────────────────────────────────────────────────────────────


---
### BASELINE — Direct LLM Call (No Gateway)

**Goal:** See what a raw LLM call looks like — no routing, no logging, no resilience.

In [23]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

In [25]:
raw_groq = ChatGroq(api_key=GROQ_API_KEY, model="openai/gpt-oss-120b", temperature=0)

section("BASELINE — Direct Groq Call")

questions = [
    "What is Kubernetes in one sentence?",
    "What is Intel SRIOV?",
]

for q in questions:
    t0 = time.time()
    r = raw_groq.invoke([HumanMessage(content=q)])
    show(q, r.content, (time.time()-t0)*1000, label="direct Groq — no gateway")



BASELINE — Direct Groq Call

──────────────────────────────────────────────────────────────
Q: What is Kubernetes in one sentence?
A: Kubernetes is an open‑source platform that automates the deployment, scaling, and management of containerized applications across clusters of machines.
⏱  2527ms | direct Groq — no gateway
──────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────
Q: What is Intel SRIOV?
A: **Intel SR‑IOV (Single‑Root I/O Virtualization)** is a hardware‑level technology that lets a single physical network (or other PCIe) device appear as multiple independent virtual devices. Each virtual device—called a **Virtual Function (VF)**—can be assigned d...
⏱  5668ms | direct Groq — no gateway
──────────────────────────────────────────────────────────────


---
# EXPERIMENT 1 — Route Through the Gateway

**Goal:** Same call, same answer — but now every request is logged in your Portkey dashboard.

**New concepts:**
- `Portkey(api_key=...)` — the gateway client
- `model="@slug/model-name"` — tells Portkey which provider to use
- `response.choices[0].message.content` — standard OpenAI response format

In [26]:
section("EXP 1 — Basic Gateway Call")

questions = [
    "What is AI and gen ai ?",
    "What is coffee and black coffee?",
]


for q in questions:
    t0 = time.time()
    r = portkey.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role":"user","content":q}]
    )
    show(q, r.choices[0].message.content, (time.time()-t0)*1000,
         label="routed via Portkey gateway")
    
print("\n✅ Check portkey.ai → Logs to see both requests fully logged!")
print("   Token count, cost, latency — all tracked. Zero extra code.")


EXP 1 — Basic Gateway Call

──────────────────────────────────────────────────────────────
Q: What is AI and gen ai ?
A: **Artificial Intelligence (AI)**  

Artificial Intelligence is a broad field of computer science that aims to create systems capable of performing tasks that normally require human intelligence. Those tasks include (but aren’t limited to):

| **Capability** | ...
⏱  9153ms | routed via Portkey gateway
──────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────
Q: What is coffee and black coffee?
A: **Coffee – the beverage**

| Aspect | Details |
|--------|----------|
| **What it is** | A brewed drink made from the roasted seeds (often called “beans”) of the coffee plant *Coffea* (most commonly *Coffea arabica* or *Coffea canephora* – “robusta”). |
| **Ho...
⏱  3457ms | routed via Portkey gateway
──────────────────────────────────────────────────────────────

✅ Check portkey.ai → Logs to see both requ